# QLoRA fine-tune - ticket triage on Kaggle T4

Fine-tunes `unsloth/Llama-3.2-3B-Instruct` on the reformatted Bitext splits and
pushes the adapter plus a GGUF for Ollama re-eval to the HF Hub.

**Setup on Kaggle:**
1. Upload `kaggle_bundle.zip` as a **Kaggle Dataset**. Kaggle unzips it.
2. In this notebook: **+ Add Input** -> attach that dataset.
3. **Settings:** Accelerator = GPU T4 x2, Internet = On.
4. **Add-ons -> Secrets:** add `HF_TOKEN` with Write scope. Optional: `WANDB_API_KEY` for the W&B report.
5. Set `SRC` below to your dataset path, then **Run All**. Model, hub, and hyperparameters live in `configs/train_qlora.yaml`.

In [ ]:
SRC = "/kaggle/input/datasets/gunturuphanee/finetune"

In [ ]:
# Copy the bundle into the writable working dir, then install.
import shutil
shutil.copytree(SRC, "/kaggle/working/repo", dirs_exist_ok=True)
%cd /kaggle/working/repo
!pip install -q -r requirements-gpu.txt
!pip install -q -e .

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
token = UserSecretsClient().get_secret("HF_TOKEN")
login(token)

In [ ]:
!python scripts/train_qlora.py --config configs/train_qlora.yaml

## Step 9 - re-eval back in Ollama

The run exports a GGUF into the adapter dir. Download it, then locally:
```bash
ollama create triage-ft -f Modelfile   # Modelfile: FROM ./model.gguf
# set configs/baseline.yaml model.name: triage-ft, then:
make eval-baseline
```
Compare against `reports/baseline_llama3.2_3b.json` for the head-to-head table.